Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1:A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2:A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq"
)
model



ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E2B4DFED70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E2B4DFEC80>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain.tools import tool
@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."
model_with_tools=model.bind_tools([get_weather])


In [ ]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool{tool_call['name']}")
    print(f"Tool{tool_call['args']}")


Toolget_weather
Tool{'city': 'Boston'}


### Tool execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in Botson?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
final_response = model_with_tools.invoke(messages)
print(final_response)

# "The current weather in Boston is 72°F and sunny."

content='Great news—Botson is enjoying sunny weather right now! 🌞 Let me know if you need a forecast for later today or any other details.' additional_kwargs={'reasoning_content': 'We need to respond to user. Probably they just asked for weather. Provide a friendly reply.'} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 161, 'total_tokens': 220, 'completion_time': 0.124065335, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.006450688, 'prompt_tokens_details': None, 'queue_time': 0.213987719, 'total_time': 0.130516023}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5a93aea882', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a07be8-4eaa-75a0-b609-b5032c2f59fa-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 161, 'output_tokens': 59, 'total_tokens': 220, 'output_token_details': {'reasoning': 20}}


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Botson?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather for city Botson. Use function get_weather.', 'tool_calls': [{'id': 'fc_477ae4bb-53f6-48d1-8aed-f1f733784f82', 'function': {'arguments': '{"city":"Botson"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 127, 'total_tokens': 171, 'completion_time': 0.091778054, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.005389001, 'prompt_tokens_details': None, 'queue_time': 0.403346634, 'total_time': 0.097167055}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07be8-4ad0-7e81-a5f6-85a590a496a7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Botson'}, 'id': 'fc_477ae4bb-53f6-48d1-8aed-f1f733784f